<a href="https://colab.research.google.com/github/Mogi21/Master-Thesis-Mammography-VLM/blob/main/Dual_view_localization_and_birads_Faster_R_CNN_Baselin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""
Phase 1 Baseline Visualization Script
Thesis: Dual-View Cross-Attention Networks for Mammography Lesion Localization and BI-RADS Classification
Description: Extracts a guaranteed ground-truth lesion image, runs inference using the Phase 1 Faster R-CNN
baseline, and generates a 3-way side-by-side comparison to illustrate single-view overlapping tissue ambiguity.
"""

import pandas as pd
import zipfile
import os
import cv2
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files

def main():
    #1. Loading annotations
    csv_path = '/content/drive/MyDrive/vindr_raw_data_11/finding_annotations.csv'
    zip_path = '/content/drive/MyDrive/vindr_pngs_full.zip'

    df = pd.read_csv(csv_path)
    df_lesions = df.dropna(subset=['xmin', 'ymin', 'xmax', 'ymax'])

    #2. Locating valid test image in archive
    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = z.namelist()
        valid_img_id, target_filename = None, None

        for file_name in all_files:
            if not file_name.lower().endswith('.png'): continue
            base_id = os.path.basename(file_name).replace('.png', '').replace('.PNG', '')

            if base_id in df_lesions['image_id'].values:
                valid_img_id = base_id
                target_filename = file_name
                break

        if valid_img_id is None:
            raise ValueError("Could not find any annotated lesion images in the ZIP archive.")

        print(f"--> Target Image ID: {valid_img_id}")
        extracted_path = z.extract(target_filename, '/content/temp_img')

    gt_boxes = df_lesions[df_lesions['image_id'] == valid_img_id][['xmin', 'ymin', 'xmax', 'ymax']].values

    #3. Preprocessing image
    image = cv2.imread(extracted_path, cv2.IMREAD_GRAYSCALE)
    image = image / 255.0
    img_tensor = torch.as_tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    #4. Initializing model & loading weights
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

    # Internal box_score_thresh set to 0.0 to expose low-confidence failure modes
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None, box_score_thresh=0.0)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 36)

    model.load_state_dict(torch.load('/content/drive/MyDrive/vindr_raw_data_11/faster_rcnn_baseline.pth', map_location=device))
    model.to(device)
    model.eval()

    #5. Generating baseline predictions
    with torch.no_grad():
        predictions = model(img_tensor.to(device))

    pred_boxes = predictions[0]['boxes'].cpu().numpy()
    pred_scores = predictions[0]['scores'].cpu().numpy()
    img_np = img_tensor.squeeze().cpu().numpy()

    #6. Generating 3-way poster visualization
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    scenarios = [
        {"type": "threshold", "val": 0.15, "title": "Threshold > 0.15\n(Missed Detection)"},
        {"type": "threshold", "val": 0.02, "title": "Threshold > 0.02\n(Scattered False Positives)"},
        {"type": "top_k", "val": 3, "title": "Top 3 Predictions\n(Forced Guesses)"}
    ]

    for ax, scenario in zip(axes, scenarios):
        ax.imshow(img_np, cmap='gray')

        # Plot Ground Truth
        for box in gt_boxes:
            xmin, ymin, xmax, ymax = box
            ax.add_patch(patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                         linewidth=3.0, edgecolor='lime', facecolor='none', label='Ground Truth'))

        # Plot Predictions
        if scenario["type"] == "threshold":
            thresh = scenario["val"]
            for box, score in zip(pred_boxes, pred_scores):
                if score > thresh:
                    xmin, ymin, xmax, ymax = box
                    ax.add_patch(patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                                 linewidth=2.5, edgecolor='red', facecolor='none', label=f'Pred ({score:.3f})'))
        else:
            k = scenario["val"]
            for i in range(min(k, len(pred_scores))):
                box = pred_boxes[i]
                score = pred_scores[i]
                xmin, ymin, xmax, ymax = box
                ax.add_patch(patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                             linewidth=2.5, edgecolor='red', facecolor='none', label=f'Pred ({score:.3f})'))

        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        if by_label:
            ax.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize=12)

        ax.axis('off')
        ax.set_title(scenario["title"], fontsize=18, fontweight='bold')

    plt.tight_layout()
    output_path = '/content/detection_comparison_poster.jpg'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

    #7. Export and Downloading final asset
    files.download(output_path)

if __name__ == '__main__':
    main()

Mounted at /content/drive
Extracting original archive of PNGs to local GPU storage...

Converting remaining DICOM images from /content/drive/MyDrive/vindr_raw_data_11/Missing_DICOMs to PNG...
